In [1]:
import numpy as np
from numba import njit, prange
import matplotlib.pyplot as plt
import matplotlib.colors as clrs
from os import makedirs
from tqdm.auto import tqdm

# plt.style.use(
#     {
#         "text.usetex": True,
#         # "text.latex.preamble": r"\usepackage{amsmath}\usepackage{amssymb}",
#         "font.family": "serif",
#         "font.serif": "Computer Modern",
#         "savefig.bbox": "tight",
#     }
# )
plt.style.use("default")

In [ ]:
@njit
def map_loop(
    x0: float,
    base_loop: np.ndarray[tuple[int], np.dtype[np.float64]],
    loop_iters: int = 1,
) -> tuple[float, float]:
    x = x0
    N = 0
    logs = np.empty(base_loop.shape[0] * loop_iters, np.float64)
    for _ in range(loop_iters):
        for base in base_loop:
            x *= base * (1.0 - x)
            if not np.isfinite(x):
                print("infinite X!")

            log = np.log(np.abs(base * (1.0 - 2 * x)))
            if np.isfinite(log):
                logs[N] = log
                N += 1
            # summation += log

    if N > base_loop.shape[0] * 2:
        lyap_exp = np.nanmean(logs[:N])
    else:
        lyap_exp = np.nan

    return x, lyap_exp


@njit(parallel=True)
def map_grid(
    x0: float,
    loop_iters: int,
    bool_loop: np.ndarray[tuple[int], np.dtype[np.bool]],
    Ngrid: int,
    arange: tuple[float, float] = (0.0, 4.0),
    brange: tuple[float, float] = (0.0, 4.0),
):
    out_x = np.empty((Ngrid, Ngrid), np.float64)
    out_lyap = np.empty((Ngrid, Ngrid), np.float64)

    amin, amax = arange
    bmin, bmax = brange
    assert amax > amin
    assert bmax > bmin
    assert Ngrid > 2

    da = (amax - amin) / (Ngrid + 1)
    db = (bmax - bmin) / (Ngrid + 1)

    A_inds = np.where(~bool_loop)[0]
    B_inds = np.where(bool_loop)[0]

    for i in prange(Ngrid):
        float_loop = np.empty((bool_loop.shape[0],), np.float64)
        a = amin + i * da
        for ind in A_inds:
            float_loop[ind] = a
        for j in range(Ngrid):
            b = bmin + j * db
            for ind in B_inds:
                float_loop[ind] = b
            out_x[j, i], out_lyap[j, i] = map_loop(x0, float_loop, loop_iters)

    return out_x, out_lyap

In [ ]:
code = "BBBBBBAAAAAA"
bools = np.array([c == "B" for c in code])

A = (3.4, 4.0)
B = (2.5, 3.4)
# Ngrid = 5000
Ngrid = 1000
out_x, out_lyap = map_grid(0.5, 75, bools, Ngrid, A, B)

In [ ]:
# plt.close("all")


# nrm = clrs.AsinhNorm(0.03, vmax=0)
# cmap = "inferno"
# fig, ax = plt.subplots(figsize=(6, 6))
# im = ax.imshow(
#     out_lyap.T[::10, ::10],
#     extent=(*A, *B),
#     norm=nrm,
#     cmap=cmap,
#     interpolation="bilinear",
#     origin="lower",
# )
# # ax.invert_yaxis()
# ax.set_aspect("auto")

# ax.set(xlabel="$a$", ylabel="$b$")
# ax.set(title=f"Lyapunov Fractal for {code}\n$x_{{n+1}}=rx_{{n}}(1-x_{{n}})$")
# # fig.colorbar(im)
# plt.show()

In [ ]:
# makedirs(code, exist_ok=True)

# for alph in tqdm([0.001, 0.005, 0.01, 0.025, 0.05, 0.25, 0.5]):
#     norm = clrs.AsinhNorm(alph, vmax=0)
#     normed = norm(out_lyap.T)
#     for cmap_name in ["viridis", "inferno", "plasma", "magma"]:
#         cmap = plt.get_cmap(cmap_name)

#         rgba_image = cmap(normed)
#         plt.imsave(
#             f"{code}/{cmap_name}_{alph:.3f}.png",
#             rgba_image,
#             origin="lower",
#             dpi=300,
#         )